In [ ]:
# import libraries
import os
import pandas as pd
import numpy as np
import seaborn as sns
sns.set(color_codes=True)
import matplotlib.pyplot as plt
%matplotlib inline
import pickle
from tensorflow import keras
import matplotlib.pyplot as plt
import numpy as np
import math
import tensorflow as tf
from numpy.random import seed
from tensorflow.random import set_seed

# set random seedindexes_AC1
seed(10)
set_seed(10)

training_dataset_name = "run53_mix_mega_shared"
testing_dataset_name = "run57_mix_mega_shared"

val_number = 6000

# If not GPU is available, set the following to -1
os.environ["CUDA_VISIBLE_DEVICES"]="0"
number_of_detectors = 6
data_path = "/data/test_newrepo/"
normalize= 1
plot=1

In [ ]:
print("Num GPUs Available: ", len(tf.config.list_physical_devices('GPU')))

In [ ]:

# load training dataseta
file_path = data_path+'/'+training_dataset_name+'_dataset.pkl'

with open(file_path, 'rb') as file:
    training_dataset_array = pickle.load(file)

# load testing dataset
file_path = data_path+'/'+testing_dataset_name+'_dataset.pkl'

with open(file_path, 'rb') as file:
    testing_dataset_array = pickle.load(file)

In [ ]:
def diff_phi(a1, a2):
    diff = abs(a1 - a2)
    if diff > 180:
        diff = 360 - diff
    return diff

def convert_degrees_to_sin_cos(degrees):
    radians = np.deg2rad(degrees)  # Converti gradi in radianti
    sin = np.sin(radians)
    cos = np.cos(radians)
    return sin, cos

def degrees_from_sin_cos(sin_value, cos_value):
    # Utilizza arctan2 per ottenere l'angolo nel giusto quadrante
    angle_rad = np.arctan2(sin_value, cos_value)
    # Converti radianti in gradi
    angle_deg = np.degrees(angle_rad)
    # Assicurati che l'angolo sia compreso tra 0 e 360 gradi
    if angle_deg < 0:
        angle_deg += 360
    return angle_deg

def convert_degress_to_sin_cos_math(degrees):
    
    rad = math.radians(degrees)

    # Calcolare il seno e il coseno dell'angolo
    sin = math.sin(rad)
    cos = math.cos(rad)

    print("Seno:", sin)
    print("Coseno:", cos)


def angular_distance(theta1, phi1, theta2, phi2):
    """
    Compute the angular (great-circle) distance in degrees between two points
    specified by (theta, phi) coordinates in degrees.

    Parameters:
        theta1, phi1: floats, coordinates of the first point (degrees).
        theta2, phi2: floats, coordinates of the second point (degrees).

    Returns:
        angular_dist_deg: float, angular distance between the two points in degrees.
    """
    # Adjust theta values by shifting -90
    theta1 = 90 - theta1
    theta2 = 90 - theta2

    # Convert angles to radians
    theta1_rad = math.radians(theta1)
    phi1_rad = math.radians(phi1)
    theta2_rad = math.radians(theta2)
    phi2_rad = math.radians(phi2)
    
    # Compute the difference in longitude
    delta_phi = abs(phi1_rad - phi2_rad)

    # Apply spherical law of cosines
    value = (math.sin(theta1_rad) * math.sin(theta2_rad) +
             math.cos(theta1_rad) * math.cos(theta2_rad) * math.cos(delta_phi))
    
    # Clamp value to [-1, 1] to avoid floating point errors in acos
    value_clamped = max(-1.0, min(1.0, value)) 
    
    angular_dist = math.acos(value_clamped)
    
    # Convert angular distance from radians to degrees
    angular_dist_deg = math.degrees(angular_dist)
    
    return angular_dist_deg
    

def l2_normalize(data):
  """
  Normalize a NumPy array using the L2 (Euclidean) norm.

  Args:
    data (numpy.ndarray): The array to normalize. Can be a 1D vector
                         or a 2D matrix (where each row is a vector to normalize).

  Returns:
    numpy.ndarray: The L2-normalized array.
  """

  data = np.array(data)
  if data.ndim == 1:
    # Case: 1D vector
    norm = np.sqrt(np.sum(data**2))
    if norm == 0:
      return data  # Avoid division by zero if the vector is null
    return data / norm
  elif data.ndim == 2:
    # Case: 2D matrix (normalize each row)
    norms = np.sqrt(np.sum(data**2, axis=1, keepdims=True))
    # Handle the case of null norms (zero rows)
    norms[norms == 0] = 1
    return data / norms
  else:
    raise ValueError("Input must be a 1D or 2D array.")

    
def prepare_dataset(data_array):

    dataset = np.empty((len(data_array),number_of_detectors))
    labels = np.empty((len(data_array),2))
    
    count = -1
    for element in data_array:
        
    
        count+=1
        
        dataset[count] = l2_normalize(element['counts'])
        labels[count][0] = float(element['coord'][0])
        labels[count][1] = float(element['coord'][1])
        
    labels = labels[:count+1]
    dataset = dataset[:count+1]

    # Convert to radians
    coords_rad = []
    
    for theta, phi in labels:
        coords_rad.append([theta, phi])
    
    theta, phi = zip(*coords_rad)
  
    labels_cosin = np.empty((len(labels),4))
    count = -1
    for element in labels:
        
        count=count+1
        
        theta_sin,theta_cos = convert_degrees_to_sin_cos(labels[count][0])
        labels_cosin[count][0] = theta_sin
        labels_cosin[count][1] = theta_cos
        
        phi_sin,phi_cos = convert_degrees_to_sin_cos(labels[count][1])
        labels_cosin[count][2] = phi_sin
        labels_cosin[count][3] = phi_cos
    
    labels_norm = labels_cosin
    
    if normalize == 1:
    
        labels_norm = np.empty((len(labels_cosin),4))
    
        for j in range(0,len(labels_cosin)):
            labels_norm[j][0] = 2 * labels_cosin[j][0] - 1
            labels_norm[j][1] = labels_cosin[j][1]
            labels_norm[j][2] = labels_cosin[j][2]
            labels_norm[j][3] = labels_cosin[j][3]
        
    else:
        labels_norm = labels_cosin

    
   
    return dataset, labels, labels_norm, theta, phi, coords_rad


In [ ]:
# prepare training and validation dataset
dataset, labels, labels_norm, theta, phi, coords_rad = prepare_dataset(training_dataset_array)

In [ ]:
# prepare testing and validation dataset
test_dataset, test_labels_raw, test_labels_norm, test_theta, test_phi, test_coords_rad = prepare_dataset(testing_dataset_array)

In [ ]:
dataset.shape

In [ ]:
test_dataset.shape

In [ ]:
#split val and test dataset
random_indices = np.random.permutation(len(dataset))

N = dataset.shape[0]
validation_indices = random_indices[:val_number]
train_indices = np.setdiff1d(np.arange(N), validation_indices)

validation_dataset = dataset[validation_indices]
training_dataset = dataset[train_indices]

validation_labels = labels_norm[validation_indices]
training_labels = labels_norm[train_indices]

test_labels = test_labels_norm


In [ ]:
plt.hist(theta,alpha=0.5)
plt.hist(test_theta,alpha=0.5)
plt.xlabel("Theta")
plt.ylabel("Counts")

In [ ]:
plt.hist(phi,alpha=0.5)
plt.hist(test_phi,alpha=0.5)
plt.xlabel("Phi")
plt.ylabel("Counts")

In [ ]:
plt.hist(training_labels[:,0],alpha=0.5)
plt.hist(test_labels_norm[:,0],alpha=0.5)


In [ ]:
plt.hist(training_labels[:,1],alpha=0.5)
plt.hist(test_labels_norm[:,1],alpha=0.5)


In [ ]:
plt.hist(training_labels[:,2],alpha=0.5)
plt.hist(test_labels_norm[:,2],alpha=0.5)


In [ ]:
plt.hist(training_labels[:,3],alpha=0.5)
plt.hist(test_labels_norm[:,3],alpha=0.5)

In [ ]:
from tensorflow.keras import backend as K

def spherical_loss_2angles(y_true, y_pred):
    # Compute the Mean Squared Error (MSE) between targets and predictions
    mse_loss = K.mean(K.square(y_true - y_pred), axis=-1)
    
    # Extract the pairs (sinθ, cosθ) and (sinφ, cosφ) from the predictions
    sin_theta, cos_theta = y_pred[:, 0], y_pred[:, 1]
    sin_phi, cos_phi     = y_pred[:, 2], y_pred[:, 3]
    
    # Compute the L2 norm of each pair
    # (should ideally be equal to 1 if the network outputs valid sine/cosine pairs)
    norm_theta = K.sqrt(sin_theta**2 + cos_theta**2)
    norm_phi   = K.sqrt(sin_phi**2 + cos_phi**2)
    
    # Compute the penalty as the squared deviation of each norm from 1
    # This encourages the network to output normalized sine/cosine pairs
    penalty_theta = K.square(norm_theta - 1.0)
    penalty_phi   = K.square(norm_phi - 1.0)
    
    penalty = penalty_theta + penalty_phi
    
    # Weight of the regularization term (can be tuned as a hyperparameter)
    alpha = 0.01
    
    # Final loss = MSE + weighted normalization penalty
    return mse_loss + alpha * penalty

In [ ]:
# Define the model architecture
model = keras.Sequential([
    keras.layers.Dense(128*2, input_shape=(number_of_detectors,)),
    keras.layers.LeakyReLU(alpha=0.1) ,
    keras.layers.Dropout(0.02),
    keras.layers.Dense(64*2),
    keras.layers.LeakyReLU(alpha=0.1),
    keras.layers.Dropout(0.02),
    keras.layers.Dense(32*2), 
    keras.layers.LeakyReLU(alpha=0.1),
    keras.layers.Dropout(0.02),
    keras.layers.Dense(4,'tanh'), 

])

# Compile the model
model.compile(optimizer=keras.optimizers.Adam(learning_rate=0.001), loss=spherical_loss_2angles)
model.summary()



In [ ]:
history = model.fit(
    x=training_dataset,
    y=training_labels,
    epochs=2000,
    batch_size=256,
    validation_data=(validation_dataset,validation_labels),
    shuffle=True,
    callbacks=[
        keras.callbacks.EarlyStopping(monitor="val_loss", patience=5, mode="min")
    ],
)


In [ ]:
#save model
save_data = False
model_name = training_dataset_name+"_tanh.keras"
if(save_data):
    model.save(data_path+"/"+model_name)

In [ ]:
fig, ax1 = plt.subplots(1, 1,figsize=(10,5))
fig.suptitle('Training')
      
ax1.plot(history.history["loss"], label="Training Loss")
ax1.plot(history.history["val_loss"], label="Validation Loss")
    
ax1.legend()
plt.show() 

if(save_data):
    
    training_results = {'loss': history.history["loss"], 'val_loss': history.history["val_loss"]}

    with open(data_path+"/"+model_name+"_training_results.pkl", "wb") as f:
        pickle.dump(training_results, f)

In [ ]:

# Evaluate the testing dataset
pred_data = model.predict(test_dataset)  

In [ ]:
import math

pred_data_renorm = np.empty((len(pred_data),4))
test_data_renorm = np.empty((len(test_labels),4))

    
for j in range(0,len(pred_data)):
    if normalize == 1:
        pred_data_renorm[j][0]  = (pred_data[j][0] + 1) / 2
    else:
        pred_data_renorm[j][0] = pred_data[j][0]
    pred_data_renorm[j][1] = pred_data[j][1]
    pred_data_renorm[j][2] = pred_data[j][2]
    pred_data_renorm[j][3] = pred_data[j][3]
    
min_val = 0
max_val = 1

for j in range(0,len(test_labels)):
    if normalize == 1:  
        test_data_renorm[j][0]  = (test_labels[j][0] + 1) / 2
    else:
        test_data_renorm[j][0] = test_labels[j][0]
    test_data_renorm[j][1] = test_labels[j][1]
    test_data_renorm[j][2] = test_labels[j][2]
    test_data_renorm[j][3] = test_labels[j][3]
      
pred_data_original = np.empty((len(pred_data),2))
test_labels_original = np.empty((len(test_labels),2))
        
for i, element in enumerate(test_data_renorm):
    test_labels_original[i][0]=degrees_from_sin_cos(element[0],element[1])
    test_labels_original[i][1]=degrees_from_sin_cos(element[2],element[3])
    

for i, element in enumerate(pred_data_renorm):
    
    pred_data_original[i][0]=degrees_from_sin_cos(element[0],element[1])
    pred_data_original[i][1]=degrees_from_sin_cos(element[2],element[3])
    
absolute_diffs_theta = np.abs(pred_data_original[:, 0] - test_labels_original[:, 0])
absolute_diffs_phi = np.abs(pred_data_original[:, 1] - test_labels_original[:, 1])

# Calculate the Mean Absolute Error (MAE) for each element separately
mae_theta = np.mean(absolute_diffs_theta)
mae_phi = np.mean(absolute_diffs_phi)

print("Mean Absolute Error for Theta:", mae_theta)
print("Mean Absolute Error for Phi:", mae_phi)

In [ ]:
fig = plt.figure(figsize=(8, 6))
plt.hist(test_labels_original[:, 0],bins=50,alpha=0.6,color="b",label="test coords")
plt.hist(pred_data_original[:,0],bins=50,alpha=0.6,color="r",label="reco coords")
plt.xlabel("Theta")
plt.legend()
plt.ylabel("Counts")


In [ ]:
fig = plt.figure(figsize=(8, 6))
plt.hist(test_labels_original[:, 1],bins=50,alpha=0.6,color="b",label="reco coords")
plt.hist(pred_data_original[:,1],bins=50,alpha=0.6,color="r",label="test coords")
plt.xlabel("Phi")
plt.legend()
plt.ylabel("Counts")

In [ ]:
fig = plt.figure(figsize=(8, 6))
plt.hist(test_labels[:,0],bins=50,alpha=0.6,color="b",label="test coords")
plt.hist(pred_data[:, 0],bins=50,alpha=0.6,color="r",label="reco coords")
plt.xlabel("Theta sin")
plt.legend()
plt.ylabel("Counts")


In [ ]:
fig = plt.figure(figsize=(8, 6))
plt.hist(test_labels[:,1],bins=50,alpha=0.6,color="b",label="test coords")
plt.hist(pred_data[:, 1],bins=50,alpha=0.6,color="r",label="reco coords")
plt.xlabel("Theta cos")
plt.legend()
plt.ylabel("Counts")


In [ ]:
fig = plt.figure(figsize=(8, 6))
plt.hist(test_labels[:,2],bins=50,alpha=0.6,color="b",label="test coords")
plt.hist(pred_data[:, 2],bins=50,alpha=0.6,color="r",label="reco coords")
plt.xlabel("Phi sin")
plt.legend()
plt.ylabel("Counts")


In [ ]:
fig = plt.figure(figsize=(8, 6))
plt.hist(test_labels[:,3],bins=50,alpha=0.6,color="b",label="test coords")
plt.hist(pred_data[:, 3],bins=50,alpha=0.6,color="r",label="reco coords")
plt.xlabel("Phi cos")
plt.legend()
plt.ylabel("Counts")


In [ ]:
# Calculate distances between corresponding coordinates

distances = []
theta_distances = []
phi_distances = []

for i in range(0,len(pred_data_original)):

    d = angular_distance(pred_data_original[i][0],pred_data_original[i][1],test_labels_original[i][0],test_labels_original[i][1])
    theta_dist = np.abs(pred_data_original[i][0]-test_labels_original[i][0])
    phi_dist = diff_phi(pred_data_original[i][1],test_labels_original[i][1])
    distances.append(d)
    theta_distances.append(theta_dist)
    phi_distances.append(phi_dist)


In [ ]:
import pickle
if save_data:

    # Salva l'array in un file usando pickle
    with open(data_path+"/"+evaluation_file+"_distances_tanh_nobkg.pkl", "wb") as f:
        pickle.dump(distances, f)

    # Salva l'array in un file usando pickle
    with open(data_path+"/"+evaluation_file+"_conf_area_tanh_nobkg.pkl", "wb") as f:
        pickle.dump(conf_area, f)


In [ ]:
import numpy as np
import healpy as hp
import matplotlib.pyplot as plt

nside = 32
npix = hp.nside2npix(nside)


m = np.array(distances)

hp.projview(
    m,
    coord=["G"],
    graticule=True,
    graticule_labels=True,
    unit="cbar label",
    xlabel="longitude",
    ylabel="latitude",
    cb_orientation="vertical",
    latitude_grid_spacing=30,
    projection_type="aitoff",
    title="Aitoff projection",
    cmap="turbo",
    nest=True
)

plt.show()


In [ ]:
print(np.mean(distances))
print(np.mean(theta_distances))
print(np.mean(phi_distances))

In [ ]:
# Definisci gli intervalli di 5 gradi
intervalli = np.arange(0, 185, 5)

# Raggruppa i conteggi in base agli intervalli di 5 gradi
conteggi_raggruppati = np.zeros((len(intervalli)))
conteggi = np.zeros((len(intervalli)))

tot_count = 0
for theta,phi,conteggio in zip(test_labels_original[:,0],test_labels_original[:,1], distances):
    
    if(True): #(phi>125 and phi<145) or 
        tot_count +=1
        indice_intervallo = int(theta // 5)
        conteggi_raggruppati[indice_intervallo] = conteggi_raggruppati[indice_intervallo]+conteggio
        conteggi[indice_intervallo]+=1

# Calcola la media dei conteggi in ciascun intervallo
conteggi_raggruppati = conteggi_raggruppati[:tot_count]
conteggi = conteggi[:tot_count]

# Visualizza l'istogramma
plt.figure(figsize=(10, 6))
plt.bar(intervalli[:-1], conteggi_raggruppati[:-1]/conteggi[:-1], width=5, align='edge',alpha=1,label="")
plt.xlabel('Theta (°)', fontsize=15)
plt.ylabel('Loc. error (°)', fontsize=15)
plt.tick_params(axis='both', labelsize=15) 
plt.title('')
plt.grid(True)
#plt.legend()
plt.show()

In [ ]:
if(save_data):

    training_histo = {'test_labels_original': test_labels_original[:,0], 'test_labels_original': test_labels_original[:,1],"distances":distances}

    with open(data_path+"/"+model_name+"_training_histo_tanh_nobkg.pkl", "wb") as f:
        pickle.dump(training_histo, f)

In [ ]:

if save_data:
    data_to_store ={"test_labels_0":test_labels_original[:,0],"test_labels_1":test_labels_original[:,1]}
    
    # Salva l'array in un file usando pickle
    with open(data_path+"/"+evaluation_file+"_hist_data_tanh_nobkg.pkl", "wb") as f:
        pickle.dump(data_to_store, f)

In [ ]:
step = 10

# Definisci gli intervalli di 5 gradi
intervalli = np.arange(0, 365, step)

# Raggruppa i conteggi in base agli intervalli di 5 gradi
conteggi_raggruppati_1 = np.zeros((len(intervalli)))
conteggi_1 = np.zeros((len(intervalli)))

conteggi_raggruppati_2 = np.zeros((len(intervalli)))
conteggi_2 = np.zeros((len(intervalli)))

conteggi_raggruppati_3 = np.zeros((len(intervalli)))
conteggi_3 = np.zeros((len(intervalli)))

conteggi_raggruppati_4 = np.zeros((len(intervalli)))
conteggi_4 = np.zeros((len(intervalli)))

tot_count_1 = 0
tot_count_2 = 0
tot_count_3 = 0
tot_count_4 = 0

for theta,phi,conteggio in zip(test_labels_original[:,0],test_labels_original[:,1], distances):
    
    if(theta>20 and theta<55):
        tot_count_1 +=1
        indice_intervallo = int(phi // step)
        conteggi_raggruppati_1[indice_intervallo] = conteggi_raggruppati_1[indice_intervallo]+conteggio
        conteggi_1[indice_intervallo]+=1

    if(theta>55 and theta<150):
        tot_count_2 +=1
        indice_intervallo = int(phi // step)
        conteggi_raggruppati_2[indice_intervallo] = conteggi_raggruppati_2[indice_intervallo]+conteggio
        conteggi_2[indice_intervallo]+=1

    if(theta>150 and theta<180):
        tot_count_3 +=1
        indice_intervallo = int(phi // step)
        conteggi_raggruppati_3[indice_intervallo] = conteggi_raggruppati_3[indice_intervallo]+conteggio
        conteggi_3[indice_intervallo]+=1

# Calcola la media dei conteggi in ciascun intervallo
conteggi_raggruppati_1 = conteggi_raggruppati_1[:tot_count_1]
conteggi_1 = conteggi_1[:tot_count_1]

conteggi_raggruppati_2 = conteggi_raggruppati_2[:tot_count_2]
conteggi_2 = conteggi_2[:tot_count_2]

conteggi_raggruppati_3 = conteggi_raggruppati_3[:tot_count_3]
conteggi_3 = conteggi_3[:tot_count_3]

# Visualizza l'istogramma
plt.figure(figsize=(10, 6))
plt.bar(intervalli[:-1], conteggi_raggruppati_1[:-1]/conteggi_1[:-1], width=step, align='edge',alpha=0.5,label="loc. error theta=[20°,55°]")

plt.bar(intervalli[:-1], conteggi_raggruppati_2[:-1]/conteggi_2[:-1], width=step, align='edge',alpha=0.5,label="loc. error theta=[55°,150°]")

plt.bar(intervalli[:-1], conteggi_raggruppati_3[:-1]/conteggi_3[:-1], width=step, align='edge',alpha=0.5,label="loc. error theta=[150°,180°]")
plt.xlabel('Phi (°)')
plt.ylabel('Loc. error')
plt.title('Loc. error in function of phi')
plt.grid(True)
plt.legend()
plt.show()